In [ ]:
%pip install -U pandas numpy pymatgen tqdm

In [ ]:
import os
import re
import ast
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

from pymatgen.core import Composition


BASE_DIR = Path("sodium_cathode_cde_matching")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "outputs"
AUDIT_DIR = BASE_DIR / "audit"

for folder in [BASE_DIR, INPUT_DIR, OUTPUT_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE_DIR.resolve())

In [ ]:
def find_first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(
        "None of these files were found:\n" + "\n".join(str(x) for x in candidates)
    )


MP_FINAL_PATH = find_first_existing([
    Path("mp_na_electrodes_conservative_candidates_v1.csv"),
    Path("mp_sodium_battery_dataset/final/mp_na_electrodes_conservative_candidates_v1.csv"),
    Path("/mnt/data/mp_na_electrodes_conservative_candidates_v1.csv"),
])

MP_RANKED_PATH = find_first_existing([
    Path("mp_na_electrodes_ranked_preliminary.csv"),
    Path("mp_sodium_battery_dataset/corrected/mp_na_electrodes_ranked_preliminary.csv"),
    Path("/mnt/data/mp_na_electrodes_ranked_preliminary.csv"),
])

BATTERY_ZIP_PATH = find_first_existing([
    Path("Battery Data.zip"),
    Path("battery_data.zip"),
    Path("data/raw/Battery Data.zip"),
    Path("/mnt/data/Battery Data.zip"),
])

MP_METADATA_PATH = find_first_existing([
    Path("materials_project_download_metadata.json"),
    Path("mp_sodium_battery_dataset/metadata/materials_project_download_metadata.json"),
    Path("/mnt/data/materials_project_download_metadata.json"),
])

print("MP final candidate file:", MP_FINAL_PATH)
print("MP ranked file:", MP_RANKED_PATH)
print("CDE battery zip:", BATTERY_ZIP_PATH)
print("MP metadata:", MP_METADATA_PATH)

In [ ]:
df_mp = pd.read_csv(MP_FINAL_PATH)
df_mp_ranked = pd.read_csv(MP_RANKED_PATH)

with open(MP_METADATA_PATH, "r", encoding="utf-8") as f:
    mp_metadata = json.load(f)

print("MP final shape:", df_mp.shape)
print("MP ranked shape:", df_mp_ranked.shape)
print("MP database version:", mp_metadata.get("mp_database_version"))

for col in [
    "basic_screening_pass",
    "hard_exclusion_free_candidate",
    "conservative_earth_abundant_candidate",
    "preliminary_family",
]:
    if col in df_mp.columns:
        print("\n", col)
        display(df_mp[col].value_counts(dropna=False).to_frame("count"))

display(df_mp.head())

In [ ]:
with zipfile.ZipFile(BATTERY_ZIP_PATH) as zf:
    print("Files inside zip:")
    for name in zf.namelist():
        print(" -", name)

    with zf.open("battery_merged.csv") as f:
        df_cde = pd.read_csv(f)

print("CDE merged database shape:", df_cde.shape)
print("Columns:")
print(df_cde.columns.tolist())

display(df_cde.head())

In [ ]:
def formula_to_reduced(formula):
    """
    Convert formula string to pymatgen reduced formula.
    Returns None if parsing fails.
    """
    if pd.isna(formula):
        return None

    s = str(formula).strip()

    if not s or s.lower() in {"nan", "none", "null"}:
        return None

    try:
        return Composition(s).reduced_formula
    except Exception:
        return None


def formula_elements(formula):
    """
    Return element symbols from formula.
    """
    if pd.isna(formula):
        return set()

    try:
        return set(el.symbol for el in Composition(str(formula)).elements)
    except Exception:
        return set()


def safe_literal_eval(x):
    """
    Safely parse CDE Extracted_name field, which is usually a stringified list of dictionaries.
    """
    if pd.isna(x):
        return None

    try:
        return ast.literal_eval(str(x))
    except Exception:
        return None


def compdict_to_reduced(d):
    """
    Convert CDE dictionary like {'Na': '3.0', 'V': '2.0', 'P': '3.0', 'O': '12.0'}
    into reduced formula.
    """
    try:
        comp = {}

        for k, v in d.items():
            if k is None:
                continue

            val = float(v)

            if val > 0:
                comp[str(k)] = val

        if not comp:
            return None

        return Composition(comp).reduced_formula

    except Exception:
        return None


FORMULA_TOKEN_RE = re.compile(
    r"([A-Z][a-z]?(?:\d*\.?\d*)?(?:[A-Z][a-z]?(?:\d*\.?\d*)?)+(?:\([A-Za-z0-9.]+\)\d*\.?\d*)*)"
)


def extract_formula_candidates_from_name(name):
    """
    Try to extract formula-like strings from the CDE Name field.
    This is supplementary because CDE Name can contain abbreviations such as NVP/C.
    """
    if pd.isna(name):
        return []

    text = str(name)
    parts = re.split(r"[/@·,;\s]+", text)

    candidates = []

    for part in parts + FORMULA_TOKEN_RE.findall(text):
        part = str(part).strip().strip("()[]{}")

        if not part:
            continue

        red = formula_to_reduced(part)

        if red is not None:
            candidates.append(red)

    # Preserve order and remove duplicates
    unique = []
    for f in candidates:
        if f not in unique:
            unique.append(f)

    return unique


def extract_cde_formulas(row):
    """
    Extract all formula candidates from CDE row using:
    1. Extracted_name
    2. Name field fallback
    """
    formulas = []

    parsed = safe_literal_eval(row.get("Extracted_name"))

    if isinstance(parsed, list):
        combined = {}

        for item in parsed:
            if isinstance(item, dict):
                red = compdict_to_reduced(item)

                if red:
                    formulas.append(red)

                # Also store combined composite formula as a weak fallback
                try:
                    for k, v in item.items():
                        combined[k] = combined.get(k, 0) + float(v)
                except Exception:
                    pass

        red_combined = compdict_to_reduced(combined) if combined else None

        if red_combined:
            formulas.append(red_combined)

    formulas.extend(extract_formula_candidates_from_name(row.get("Name")))

    unique = []
    for f in formulas:
        if f and f not in unique:
            unique.append(f)

    return unique


def split_doi_list(x):
    if pd.isna(x):
        return []

    return [
        d.strip()
        for d in str(x).split(",")
        if d.strip() and d.strip().lower() != "nan"
    ]


def has_warning_flag(x, flags):
    if pd.isna(x):
        return False

    text = str(x).upper()

    return any(flag.upper() in text for flag in flags)

In [ ]:
required_cols = [
    "Property",
    "Name",
    "Value",
    "Unit",
    "Raw_value",
    "Raw_unit",
    "Warning",
    "Type",
    "DOI",
    "Title",
    "Journal",
    "Date",
    "Tag",
    "Info",
    "Specifier",
    "Num_records",
    "Extracted_name",
]

for col in required_cols:
    if col not in df_cde.columns:
        df_cde[col] = np.nan

df_cde["Value"] = pd.to_numeric(df_cde["Value"], errors="coerce")
df_cde["Num_records"] = pd.to_numeric(df_cde["Num_records"], errors="coerce").fillna(1)

df_cde["property_clean"] = df_cde["Property"].astype(str).str.strip()

df_cde["warning_has_L_or_R"] = df_cde["Warning"].map(
    lambda x: has_warning_flag(x, ["L", "R"])
)

df_cde["warning_has_S"] = df_cde["Warning"].map(
    lambda x: has_warning_flag(x, ["S"])
)

df_cde["warning_clean_no_LR"] = ~df_cde["warning_has_L_or_R"]

context_cols = ["Name", "Title", "Type", "Specifier"]

combined_context = (
    df_cde[context_cols]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.lower()
)

df_cde["sodium_text_context"] = combined_context.str.contains(
    r"\bna\b|sodium|sodium[- ]ion|na[- ]ion",
    regex=True,
)

df_cde["cathode_text_context"] = combined_context.str.contains(
    r"cathode|positive electrode",
    regex=True,
)

df_cde["anode_or_electrolyte_context"] = combined_context.str.contains(
    r"anode|negative electrode|electrolyte|separator",
    regex=True,
)

print("CDE rows:", len(df_cde))
print("Warning L/R rows:", int(df_cde["warning_has_L_or_R"].sum()))
print("Warning S rows:", int(df_cde["warning_has_S"].sum()))
print("Sodium text-context rows:", int(df_cde["sodium_text_context"].sum()))
print("Cathode text-context rows:", int(df_cde["cathode_text_context"].sum()))

display(df_cde["property_clean"].value_counts().to_frame("count"))

In [ ]:
tqdm.pandas()

df_cde["cde_formula_candidates"] = df_cde.progress_apply(
    extract_cde_formulas,
    axis=1,
)

df_cde["formula_candidate_count"] = df_cde["cde_formula_candidates"].map(len)

print("Rows with at least one parsed formula:", int((df_cde["formula_candidate_count"] > 0).sum()))
print("Rows without parsed formula:", int((df_cde["formula_candidate_count"] == 0).sum()))

display(df_cde[["Name", "Extracted_name", "cde_formula_candidates"]].head(20))

In [ ]:
keep_cols = [
    "Property",
    "property_clean",
    "Name",
    "Value",
    "Unit",
    "Raw_value",
    "Raw_unit",
    "Warning",
    "Type",
    "DOI",
    "Title",
    "Journal",
    "Date",
    "Tag",
    "Info",
    "Specifier",
    "Num_records",
    "warning_has_L_or_R",
    "warning_has_S",
    "warning_clean_no_LR",
    "sodium_text_context",
    "cathode_text_context",
    "anode_or_electrolyte_context",
    "cde_formula_candidates",
]

df_cde_formula_records = (
    df_cde[keep_cols]
    .explode("cde_formula_candidates")
    .rename(columns={"cde_formula_candidates": "cde_reduced_formula"})
)

df_cde_formula_records = df_cde_formula_records[
    df_cde_formula_records["cde_reduced_formula"].notna()
].copy()

df_cde_formula_records["cde_formula_elements"] = df_cde_formula_records[
    "cde_reduced_formula"
].map(lambda f: ",".join(sorted(formula_elements(f))))

df_cde_formula_records["cde_formula_contains_Na"] = df_cde_formula_records[
    "cde_formula_elements"
].map(lambda s: "Na" in str(s).split(","))

df_cde_formula_records["cde_sodium_evidence"] = (
    df_cde_formula_records["cde_formula_contains_Na"]
    | df_cde_formula_records["sodium_text_context"]
)

df_cde_formula_records["cde_cathode_likely"] = (
    df_cde_formula_records["cathode_text_context"]
    | (
        df_cde_formula_records["cde_formula_contains_Na"]
        & ~df_cde_formula_records["anode_or_electrolyte_context"]
    )
)

df_cde_formula_records["doi_list"] = df_cde_formula_records["DOI"].map(split_doi_list)

print("Exploded CDE formula records:", df_cde_formula_records.shape)
print("Formula records containing Na:", int(df_cde_formula_records["cde_formula_contains_Na"].sum()))
print("Formula records with sodium evidence:", int(df_cde_formula_records["cde_sodium_evidence"].sum()))
print("Formula records likely cathode:", int(df_cde_formula_records["cde_cathode_likely"].sum()))

display(df_cde_formula_records.head())

In [ ]:
VALID_PROPERTIES = [
    "Capacity",
    "Voltage",
    "Energy",
    "Coulombic Efficiency",
    "Conductivity",
]

df_cde_formula_clean = df_cde_formula_records[
    df_cde_formula_records["property_clean"].isin(VALID_PROPERTIES)
    & df_cde_formula_records["Value"].notna()
    & df_cde_formula_records["warning_clean_no_LR"]
].copy()

df_cde_na_cathode_clean = df_cde_formula_clean[
    df_cde_formula_clean["cde_sodium_evidence"]
    & df_cde_formula_clean["cde_cathode_likely"]
].copy()

cde_clean_path = OUTPUT_DIR / "cde_formula_records_clean_no_LR.csv"
cde_na_cathode_path = OUTPUT_DIR / "cde_na_cathode_clean_filtered.csv"

df_cde_formula_clean.to_csv(cde_clean_path, index=False)
df_cde_na_cathode_clean.to_csv(cde_na_cathode_path, index=False)

print("Saved:", cde_clean_path)
print("Saved:", cde_na_cathode_path)

print("Clean formula records, no L/R:", df_cde_formula_clean.shape)
print("Na-cathode clean filtered records:", df_cde_na_cathode_clean.shape)

print("\nNa-cathode property counts:")
display(df_cde_na_cathode_clean["property_clean"].value_counts().to_frame("count"))

display(df_cde_na_cathode_clean.head())

In [ ]:
def n_unique_dois(series):
    unique_dois = set()

    for item in series:
        if isinstance(item, list):
            unique_dois.update(item)

    return len(unique_dois)


def join_unique_values(series, max_items=5):
    values = []

    for x in series.dropna().astype(str):
        item = x.strip()

        if item and item not in values:
            values.append(item)

        if len(values) >= max_items:
            break

    return " | ".join(values)


df_cde_formula_property_agg = (
    df_cde_formula_clean
    .groupby(["cde_reduced_formula", "property_clean"], dropna=False)
    .agg(
        cde_value_median=("Value", "median"),
        cde_value_mean=("Value", "mean"),
        cde_value_min=("Value", "min"),
        cde_value_max=("Value", "max"),
        cde_record_count=("Value", "size"),
        cde_num_records_sum=("Num_records", "sum"),
        cde_doi_count=("doi_list", n_unique_dois),
        cde_cathode_context_count=("cathode_text_context", "sum"),
        cde_sodium_text_context_count=("sodium_text_context", "sum"),
        cde_warning_S_count=("warning_has_S", "sum"),
        cde_example_names=("Name", join_unique_values),
        cde_example_titles=("Title", join_unique_values),
        cde_example_journals=("Journal", join_unique_values),
    )
    .reset_index()
)

agg_path = OUTPUT_DIR / "cde_formula_property_aggregated_clean_no_LR.csv"
df_cde_formula_property_agg.to_csv(agg_path, index=False)

print("Saved:", agg_path)
print("Aggregated formula-property rows:", df_cde_formula_property_agg.shape)

display(df_cde_formula_property_agg.head())

In [ ]:
df_mp = df_mp.copy()
df_mp["mp_index"] = range(len(df_mp))

mp_formula_key_cols = [
    "formula_discharge",
    "formula_charge",
    "framework_formula",
    "formula_for_parsing",
    "battery_formula",
]

mp_formula_key_cols = [
    col for col in mp_formula_key_cols
    if col in df_mp.columns
]

mp_key_rows = []

for _, row in df_mp.iterrows():
    for col in mp_formula_key_cols:
        original_formula = row.get(col)

        reduced_formula = formula_to_reduced(original_formula)

        if reduced_formula:
            mp_key_rows.append({
                "mp_index": row["mp_index"],
                "mp_key_source": col,
                "mp_original_formula": original_formula,
                "mp_reduced_formula": reduced_formula,
            })

df_mp_formula_keys = pd.DataFrame(mp_key_rows).drop_duplicates()

mp_keys_path = OUTPUT_DIR / "mp_formula_keys_for_cde_matching.csv"
df_mp_formula_keys.to_csv(mp_keys_path, index=False)

print("Saved:", mp_keys_path)
print("MP formula keys:", df_mp_formula_keys.shape)
print("Unique MP reduced formulas:", df_mp_formula_keys["mp_reduced_formula"].nunique())

display(df_mp_formula_keys.head(20))

In [ ]:
df_exact_matches_long = df_mp_formula_keys.merge(
    df_cde_formula_property_agg,
    left_on="mp_reduced_formula",
    right_on="cde_reduced_formula",
    how="left",
)

df_exact_matches_long = df_exact_matches_long[
    df_exact_matches_long["cde_record_count"].notna()
].copy()

exact_long_path = OUTPUT_DIR / "mp_cde_exact_formula_matches_long.csv"
df_exact_matches_long.to_csv(exact_long_path, index=False)

print("Saved:", exact_long_path)
print("Exact matched rows:", df_exact_matches_long.shape)
print("MP electrode records with exact CDE formula evidence:", df_exact_matches_long["mp_index"].nunique())

display(df_exact_matches_long.head())

In [ ]:
if len(df_exact_matches_long) > 0:

    df_by_mp_property = (
        df_exact_matches_long
        .groupby(["mp_index", "property_clean"], dropna=False)
        .agg(
            cde_value_median=("cde_value_median", "median"),
            cde_record_count=("cde_record_count", "sum"),
            cde_doi_count=("cde_doi_count", "sum"),
            cde_cathode_context_count=("cde_cathode_context_count", "sum"),
            cde_sodium_text_context_count=("cde_sodium_text_context_count", "sum"),
            cde_warning_S_count=("cde_warning_S_count", "sum"),
        )
        .reset_index()
    )

    value_pivot = df_by_mp_property.pivot(
        index="mp_index",
        columns="property_clean",
        values="cde_value_median",
    )

    record_pivot = df_by_mp_property.pivot(
        index="mp_index",
        columns="property_clean",
        values="cde_record_count",
    )

    doi_pivot = df_by_mp_property.pivot(
        index="mp_index",
        columns="property_clean",
        values="cde_doi_count",
    )

    value_pivot.columns = [
        f"cde_{str(c).lower().replace(' ', '_')}_median"
        for c in value_pivot.columns
    ]

    record_pivot.columns = [
        f"cde_{str(c).lower().replace(' ', '_')}_record_count"
        for c in record_pivot.columns
    ]

    doi_pivot.columns = [
        f"cde_{str(c).lower().replace(' ', '_')}_doi_count"
        for c in doi_pivot.columns
    ]

    df_cde_wide_by_mp = pd.concat(
        [value_pivot, record_pivot, doi_pivot],
        axis=1,
    ).reset_index()

    df_match_summary = (
        df_exact_matches_long
        .groupby("mp_index")
        .agg(
            cde_exact_formula_match_count=(
                "cde_reduced_formula",
                lambda x: len(set(x.dropna())),
            ),
            cde_matched_reduced_formulas=(
                "cde_reduced_formula",
                lambda x: ",".join(sorted(set(x.dropna()))),
            ),
            cde_matched_mp_key_sources=(
                "mp_key_source",
                lambda x: ",".join(sorted(set(x.dropna()))),
            ),
            cde_total_record_count=("cde_record_count", "sum"),
            cde_total_doi_count=("cde_doi_count", "sum"),
            cde_total_cathode_context_count=("cde_cathode_context_count", "sum"),
            cde_total_sodium_context_count=("cde_sodium_text_context_count", "sum"),
            cde_total_series_warning_count=("cde_warning_S_count", "sum"),
        )
        .reset_index()
    )

    df_cde_evidence_by_mp = df_match_summary.merge(
        df_cde_wide_by_mp,
        on="mp_index",
        how="left",
    )

else:
    df_cde_evidence_by_mp = pd.DataFrame({"mp_index": []})

print("MP-level CDE evidence table:", df_cde_evidence_by_mp.shape)
display(df_cde_evidence_by_mp.head())

In [ ]:
df_mp_master = df_mp.merge(
    df_cde_evidence_by_mp,
    on="mp_index",
    how="left",
)

df_mp_master["cde_exact_formula_match"] = (
    df_mp_master["cde_exact_formula_match_count"]
    .fillna(0)
    .astype(int)
    > 0
)

df_mp_master["cde_has_capacity_evidence"] = (
    df_mp_master.get("cde_capacity_record_count", pd.Series(index=df_mp_master.index, dtype=float))
    .fillna(0)
    > 0
)

df_mp_master["cde_has_voltage_evidence"] = (
    df_mp_master.get("cde_voltage_record_count", pd.Series(index=df_mp_master.index, dtype=float))
    .fillna(0)
    > 0
)

df_mp_master["cde_has_capacity_and_voltage_evidence"] = (
    df_mp_master["cde_has_capacity_evidence"]
    & df_mp_master["cde_has_voltage_evidence"]
)

master_path = OUTPUT_DIR / "mp_na_electrodes_master_v1_with_cde_evidence.csv"
exact_evidence_path = OUTPUT_DIR / "mp_na_electrodes_with_cde_exact_evidence.csv"

df_mp_master.to_csv(master_path, index=False)
df_mp_master[df_mp_master["cde_exact_formula_match"]].to_csv(exact_evidence_path, index=False)

print("Saved:", master_path)
print("Saved:", exact_evidence_path)

print("Total MP records:", len(df_mp_master))
print("MP records with exact CDE evidence:", int(df_mp_master["cde_exact_formula_match"].sum()))
print("MP records with CDE capacity evidence:", int(df_mp_master["cde_has_capacity_evidence"].sum()))
print("MP records with CDE voltage evidence:", int(df_mp_master["cde_has_voltage_evidence"].sum()))
print("MP records with both capacity and voltage evidence:", int(df_mp_master["cde_has_capacity_and_voltage_evidence"].sum()))

if "conservative_earth_abundant_candidate" in df_mp_master.columns:
    conservative = df_mp_master["conservative_earth_abundant_candidate"] == True
    print(
        "Conservative candidates with exact CDE evidence:",
        int((conservative & df_mp_master["cde_exact_formula_match"]).sum()),
    )

display(
    df_mp_master[
        [
            "battery_formula",
            "formula_discharge",
            "framework_formula",
            "preliminary_family",
            "average_voltage",
            "capacity_grav",
            "energy_grav",
            "conservative_earth_abundant_candidate",
            "cde_exact_formula_match",
            "cde_matched_reduced_formulas",
            "cde_capacity_median",
            "cde_voltage_median",
            "cde_capacity_record_count",
            "cde_voltage_record_count",
            "cde_total_doi_count",
        ]
    ].head(30)
)

In [ ]:
audit_rows = []

def add_metric(name, value):
    audit_rows.append({"metric": name, "value": value})


add_metric("mp_database_version", mp_metadata.get("mp_database_version"))
add_metric("mp_total_records", len(df_mp_master))
add_metric("mp_basic_screening_pass", int(df_mp_master.get("basic_screening_pass", pd.Series(dtype=bool)).fillna(False).sum()))
add_metric("mp_hard_exclusion_free_candidates", int(df_mp_master.get("hard_exclusion_free_candidate", pd.Series(dtype=bool)).fillna(False).sum()))
add_metric("mp_conservative_earth_abundant_candidates", int(df_mp_master.get("conservative_earth_abundant_candidate", pd.Series(dtype=bool)).fillna(False).sum()))

add_metric("cde_merged_rows", len(df_cde))
add_metric("cde_formula_records", len(df_cde_formula_records))
add_metric("cde_formula_records_clean_no_LR", len(df_cde_formula_clean))
add_metric("cde_na_cathode_clean_filtered_records", len(df_cde_na_cathode_clean))
add_metric("cde_aggregated_formula_property_rows", len(df_cde_formula_property_agg))

add_metric("mp_records_with_exact_cde_formula_evidence", int(df_mp_master["cde_exact_formula_match"].sum()))
add_metric("mp_records_with_cde_capacity_evidence", int(df_mp_master["cde_has_capacity_evidence"].sum()))
add_metric("mp_records_with_cde_voltage_evidence", int(df_mp_master["cde_has_voltage_evidence"].sum()))
add_metric("mp_records_with_cde_capacity_and_voltage_evidence", int(df_mp_master["cde_has_capacity_and_voltage_evidence"].sum()))

if "conservative_earth_abundant_candidate" in df_mp_master.columns:
    conservative_mask = df_mp_master["conservative_earth_abundant_candidate"] == True
    add_metric("conservative_candidates_with_exact_cde_evidence", int((conservative_mask & df_mp_master["cde_exact_formula_match"]).sum()))
    add_metric("conservative_candidates_with_cde_capacity_evidence", int((conservative_mask & df_mp_master["cde_has_capacity_evidence"]).sum()))
    add_metric("conservative_candidates_with_cde_voltage_evidence", int((conservative_mask & df_mp_master["cde_has_voltage_evidence"]).sum()))

df_audit = pd.DataFrame(audit_rows)

audit_path = AUDIT_DIR / "cde_matching_audit_summary.csv"
df_audit.to_csv(audit_path, index=False)

print("Saved:", audit_path)
display(df_audit)

In [ ]:
candidate_cols = [
    "battery_formula",
    "formula_discharge",
    "framework_formula",
    "preliminary_family",
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
    "mp_preliminary_score",
    "conservative_earth_abundant_candidate",
    "hard_exclusion_free_candidate",
    "cde_exact_formula_match",
    "cde_matched_reduced_formulas",
    "cde_capacity_median",
    "cde_voltage_median",
    "cde_energy_median",
    "cde_capacity_record_count",
    "cde_voltage_record_count",
    "cde_total_doi_count",
]

candidate_cols = [c for c in candidate_cols if c in df_mp_master.columns]

df_top_evidence = df_mp_master[
    (df_mp_master.get("conservative_earth_abundant_candidate", False) == True)
    & (df_mp_master["cde_exact_formula_match"] == True)
].copy()

sort_cols = [c for c in ["mp_preliminary_score", "cde_total_doi_count"] if c in df_top_evidence.columns]

if sort_cols:
    df_top_evidence = df_top_evidence.sort_values(
        by=sort_cols,
        ascending=[False] * len(sort_cols),
    )

top_evidence_path = OUTPUT_DIR / "top_conservative_candidates_with_cde_evidence.csv"
df_top_evidence[candidate_cols].to_csv(top_evidence_path, index=False)

print("Saved:", top_evidence_path)
print("Top conservative candidates with exact CDE evidence:", len(df_top_evidence))

display(df_top_evidence[candidate_cols].head(50))

In [ ]:
expected_outputs = [
    OUTPUT_DIR / "cde_formula_records_clean_no_LR.csv",
    OUTPUT_DIR / "cde_na_cathode_clean_filtered.csv",
    OUTPUT_DIR / "cde_formula_property_aggregated_clean_no_LR.csv",
    OUTPUT_DIR / "mp_formula_keys_for_cde_matching.csv",
    OUTPUT_DIR / "mp_cde_exact_formula_matches_long.csv",
    OUTPUT_DIR / "mp_na_electrodes_master_v1_with_cde_evidence.csv",
    OUTPUT_DIR / "mp_na_electrodes_with_cde_exact_evidence.csv",
    OUTPUT_DIR / "top_conservative_candidates_with_cde_evidence.csv",
    AUDIT_DIR / "cde_matching_audit_summary.csv",
]

print("Final output file audit:")

for path in expected_outputs:
    status = "FOUND" if path.exists() else "MISSING"
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    print(f"{status:7s} | {size_mb:8.2f} MB | {path}")